In [3]:
import numpy as np
import pandas as pd
from metrics.fix_coords import fix_coords_by_svd, fix_coords_by_xyz
from metrics.gdt_ts import gdt_ts
from metrics.tmscore import tmsocre
from metrics.rmsd import rmsd
from metrics.lddt import lddt

data = np.load('saved_data.npz', allow_pickle=True)

In [5]:
df = {
    'pdb_id': [],
    'cutoff': [],
    'align': [],
    'rmsd': [],
    'tmscore': [],
    'gdt_ts': [],
    'lddt': [],
}
for pdb_id in data.files:
    reference = np.load(f'../../../data/FormatData/{pdb_id}.npz', allow_pickle=True)['coord']
    
    for align in ['svd', 'f3kp']:
        for cutoff in data[pdb_id].item().keys():
            coord = data[pdb_id].item()[cutoff]
            coord = np.mean(coord, axis=0)
            
            if align == 'svd':
                coord = fix_coords_by_svd(reference, coord)
            else:
                coord = fix_coords_by_xyz(coord)
                reference = fix_coords_by_xyz(reference)
            
            df['pdb_id'].append(pdb_id)
            df['cutoff'].append(cutoff)
            df['align'].append(align)
            df['rmsd'].append(rmsd(reference, coord))
            df['tmscore'].append(tmsocre(reference, coord))
            df['gdt_ts'].append(gdt_ts(reference, coord))
            df['lddt'].append(lddt(reference, coord))
df = pd.DataFrame(df)
df.to_csv('metrics.csv', index=False)